In [28]:
import kagglehub;
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set(style="whitegrid", palette="muted", font_scale=1.1)

# Download latest version
path = kagglehub.dataset_download("solonofathens/10000-rows-randomly-sampled-from-lcallcsv")
# path = '/kaggle/input/10000-rows-randomly-sampled-from-lcallcsv';
print("Path to dataset files:", path)

files = os.listdir(path) 
print("Files in this folder:", files)
full_file_path = os.path.join(path, files[0])
df = pd.read_csv(full_file_path);

Using Colab cache for faster access to the '10000-rows-randomly-sampled-from-lcallcsv' dataset.
Path to dataset files: /kaggle/input/10000-rows-randomly-sampled-from-lcallcsv
Files in this folder: ['lc_10000_rows.csv']


# 1. Quick inspection

In [10]:
display(df.head());

,Unnamed: 0,X,id,load_entry_id,file_descriptor,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,...,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount
0,296192,296192,3316245,-2140141508,3c,NaN,17600,17600,17600,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,219806,219806,3855783,-2140141507,3d,NaN,3600,3600,3600,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,142772,142772,3710061,-2140141507,3d,NaN,20000,20000,20000,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,433148,433148,3471924,-2140141508,3c,NaN,12000,12000,12000,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,429639,429639,3468046,-2140141508,3c,NaN,13075,13075,13075,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df.shape);

(1000, 141)


In [4]:
df.info();

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 141 entries, Unnamed: 0 to hardship_last_payment_amount
dtypes: float64(94), int64(22), object(25)
memory usage: 1.1+ MB


In [30]:
print("\nData types:")
print(df.dtypes)


Data types:
Unnamed: 0                                      int64
X                                               int64
id                                              int64
load_entry_id                                   int64
file_descriptor                                object
                                               ...   
hardship_dpd                                  float64
hardship_loan_status                          float64
orig_projected_additional_accrued_interest    float64
hardship_payoff_balance_amount                float64
hardship_last_payment_amount                  float64
Length: 141, dtype: object


In [ ]:
df.describe(include='all') # Summary stats

,Unnamed: 0,X,id,load_entry_id,file_descriptor,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,...,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount
count,1000.000000,1000.000000,1.000000e+03,1.000000e+03,1000,0.0,1000.000000,1000.000000,1000.000000,1000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
unique,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,NaN,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,3d,NaN,NaN,NaN,NaN,36 months,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,287,NaN,NaN,NaN,NaN,782,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,370706.792000,370706.792000,3.605385e+06,-2.140142e+09,NaN,NaN,14167.300000,14167.300000,14160.850000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,216560.821123,216560.821123,3.718515e+05,1.763403e+00,NaN,NaN,8388.770526,8388.770526,8385.264154,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2243.000000,2243.000000,3.093924e+06,-2.140142e+09,NaN,NaN,1000.000000,1000.000000,1000.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,176136.750000,176136.750000,3.289652e+06,-2.140142e+09,NaN,NaN,8000.000000,8000.000000,8000.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,371380.000000,371380.000000,3.507688e+06,-2.140142e+09,NaN,NaN,12000.000000,12000.000000,12000.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,562971.750000,562971.750000,3.882713e+06,-2.140142e+09,NaN,NaN,20000.000000,20000.000000,19950.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df.isnull().sum()); # missing values

Unnamed: 0                                       0
X                                                0
id                                               0
load_entry_id                                    0
file_descriptor                                  0
                                              ... 
hardship_dpd                                  1000
hardship_loan_status                          1000
orig_projected_additional_accrued_interest    1000
hardship_payoff_balance_amount                1000
hardship_last_payment_amount                  1000
Length: 141, dtype: int64


In [ ]:
print(df.nunique()); # unique values

Unnamed: 0                                    1000
X                                             1000
id                                            1000
load_entry_id                                   10
file_descriptor                                 10
                                              ... 
hardship_dpd                                     0
hardship_loan_status                             0
orig_projected_additional_accrued_interest       0
hardship_payoff_balance_amount                   0
hardship_last_payment_amount                     0
Length: 141, dtype: int64


In [29]:
print("\nColumns:", list(df.columns))


Columns: ['Unnamed: 0', 'X', 'id', 'load_entry_id', 'file_descriptor', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 't

# 2. Cleaning basics

In [ ]:
print(df.duplicated().sum()); # duplicate rows

0


In [16]:
print(df.drop_duplicates());

     Unnamed: 0       X       id  load_entry_id file_descriptor  member_id  \
0        296192  296192  3316245    -2140141508              3c        NaN   
1        219806  219806  3855783    -2140141507              3d        NaN   
2        142772  142772  3710061    -2140141507              3d        NaN   
3        433148  433148  3471924    -2140141508              3c        NaN   
4        429639  429639  3468046    -2140141508              3c        NaN   
..          ...     ...      ...            ...             ...        ...   
995      611200  611200  3173716    -2140141509              3b        NaN   
996      455023  455023  3496047    -2140141508              3c        NaN   
997      586249  586249  3147029    -2140141509              3b        NaN   
998      195032  195032  3810868    -2140141507              3d        NaN   
999      573506  573506  3133282    -2140141509              3b        NaN   

     loan_amnt  funded_amnt  funded_amnt_inv       term  ... ha

In [ ]:
df = df.dropna(subset=['id']); # Drop rows

In [25]:
# Impute simple:
df['hardship_amount'] = df['hardship_amount'].fillna(df['hardship_amount'].median());